# Hướng dẫn Train Mô hình Đề xuất: LayoutLMv3 (Hình ảnh + Tọa độ + Text)

Mô hình này là "Trùm cuối" của Đồ án, kết hợp cả Đặc trưng Hình ảnh (ViT) thay vì chỉ dùng Chữ như bản v1.

## 🚀 Hướng dẫn:
1. Vẫn mở 1 tab Notebook mới trên Kaggle, chọn **GPU T4 x2**.
2. Vẫn dùng chung **Dataset** `avir-kie-dataset` như cũ.
3. Chạy từng ô lệnh bên dưới.

In [ ]:
!pip install -q datasets seqeval evaluate transformers torchvision

In [ ]:
import os
import shutil

# Tự động tìm kiếm thư mục dataset
found_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train' in dirs and 'val' in dirs:
        if os.path.exists(os.path.join(root, 'train', 'metadata.json')):
            found_dir = root
            break

WORKING_DIR = "/kaggle/working/synthetic_dataset"
if found_dir:
    print(f"Tìm thấy dataset gốc tại: {found_dir}")
    if not os.path.exists(WORKING_DIR):
        print("Đang copy sang ổ cứng ảo /kaggle/working/ để đọc/ghi nhanh hơn...")
        shutil.copytree(found_dir, WORKING_DIR)
        print("Xong!")
    else:
        print("Đã có sẵn dataset trong working dir.")
else:
    print("LỖI KHÔNG TÌM THẤY DATASET!")

In [ ]:
%%writefile /kaggle/working/train_layoutlmv3.py
import os
import json
import argparse
import numpy as np
from PIL import Image

os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Tránh lỗi DataParallel của HuggingFace

from datasets import Dataset, Features, Sequence, Value, ClassLabel
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)
import evaluate

LABELS = [
    "O", "B-SELLER", "I-SELLER", "B-ADDRESS", "I-ADDRESS",
    "B-TIMESTAMP", "I-TIMESTAMP", "B-TOTAL_COST", "I-TOTAL_COST",
    "B-ITEM_NAME", "I-ITEM_NAME", "B-ITEM_QTY", "I-ITEM_QTY",
    "B-ITEM_PRICE", "I-ITEM_PRICE", "B-ITEM_AMOUNT", "I-ITEM_AMOUNT",
    "B-OTHER", "I-OTHER"
]
label2id = {lbl: idx for idx, lbl in enumerate(LABELS)}
id2label = {idx: lbl for idx, lbl in enumerate(LABELS)}

def normalize_bbox(bbox, width, height):
    return [
        max(0, min(1000, int(1000 * (bbox[0] / width)))),
        max(0, min(1000, int(1000 * (bbox[1] / height)))),
        max(0, min(1000, int(1000 * (bbox[2] / width)))),
        max(0, min(1000, int(1000 * (bbox[3] / height))))
    ]

def split_box_into_words(text, box):
    words = text.split()
    if not words: return [], []
    x1, y1, x2, y2 = box
    total_len = max(1, sum(len(w) for w in words))
    word_boxes = []
    current_x = x1
    for w in words:
        w_width = (len(w) / total_len) * (x2 - x1)
        word_boxes.append([current_x, y1, current_x + w_width, y2])
        current_x += w_width + (x2 - x1)*0.02
    return words, word_boxes

def load_data_from_metadata(metadata_path, base_dir):
    with open(metadata_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    all_words, all_bboxes, all_ner_tags, all_images = [], [], [], []
    
    for item in data:
        img_file = os.path.join(base_dir, item['image_path'])
        label_file = os.path.join(base_dir, item['label_path'])
        if not os.path.exists(label_file) or not os.path.exists(img_file):
            continue
            
        with open(label_file, 'r', encoding='utf-8') as lf:
            label_data = json.load(lf)
            
        width = label_data.get('image_width', 600)
        height = label_data.get('image_height', 800)
        
        words, bboxes, ner_tags = [], [], []
        for ann in label_data.get('annotations', []):
            label = ann['label']
            text = ann['text']
            box = ann['box']
            label = str(ann['label']).upper()
            if label.startswith("ITEM_NAME"): label = "ITEM_NAME"
            elif label.startswith("ITEM_QTY"): label = "ITEM_QTY"
            elif label.startswith("ITEM_PRICE"): label = "ITEM_PRICE"
            elif label.startswith("ITEM_AMOUNT"): label = "ITEM_AMOUNT"
            elif label not in ["SELLER", "ADDRESS", "TIMESTAMP", "TOTAL_COST"]:
                label = "OTHER"
            
            w_list, box_list = split_box_into_words(text, box)
            for i, (w, b) in enumerate(zip(w_list, box_list)):
                words.append(w)
                bboxes.append(normalize_bbox(b, width, height))
                ner_tags.append(label2id[f"B-{label}"] if i == 0 else label2id[f"I-{label}"])
                
        if words:
            all_words.append(words)
            all_bboxes.append(bboxes)
            all_ner_tags.append(ner_tags)
            all_images.append(img_file)
            
    return {
        "id": list(range(len(all_words))),
        "words": all_words,
        "bboxes": all_bboxes,
        "ner_tags": all_ner_tags,
        "image_path": all_images
    }

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--smoke-test", action="store_true")
    parser.add_argument("--model-name", type=str, default="microsoft/layoutlmv3-base")
    args = parser.parse_args()

    print("Loading v3 datasets...")
    train_dict = load_data_from_metadata("/kaggle/working/synthetic_dataset/train/metadata.json", "/kaggle/working/synthetic_dataset")
    val_dict = load_data_from_metadata("/kaggle/working/synthetic_dataset/val/metadata.json", "/kaggle/working/synthetic_dataset")
    
    if args.smoke_test:
        for key in train_dict: train_dict[key] = train_dict[key][:16]
        for key in val_dict: val_dict[key] = val_dict[key][:8]

    # Tạo processor v3 (Bao gồm Tokenizer và Feature Extractor)
    processor = LayoutLMv3Processor.from_pretrained(args.model_name, apply_ocr=False)

    # Hàm xử lý data theo batch (Rất tốn RAM nên để batch_size nhỏ)
    def prepare_examples(examples):
        images = [Image.open(path).convert("RGB") for path in examples['image_path']]
        encoding = processor(
            images,
            examples["words"],
            boxes=examples["bboxes"],
            word_labels=examples["ner_tags"],
            truncation=True,
            padding="max_length",
            max_length=512
        )
        return encoding

    train_dataset = Dataset.from_dict(train_dict)
    val_dataset = Dataset.from_dict(val_dict)
    
    print("Đang nạp và xử lý hình ảnh (có thể tốn vài phút)...")
    train_tokenized = train_dataset.map(
        prepare_examples, 
        batched=True, 
        batch_size=8, # Tránh tràn RAM của Kaggle
        remove_columns=train_dataset.column_names
    )
    val_tokenized = val_dataset.map(
        prepare_examples, 
        batched=True, 
        batch_size=8,
        remove_columns=val_dataset.column_names
    )

    print("Loading v3 model...")
    model = LayoutLMv3ForTokenClassification.from_pretrained(
        args.model_name,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id
    )
    
    metric = evaluate.load("seqeval")
    def compute_metrics(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)
        true_predictions = [[LABELS[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]
        true_labels = [[LABELS[l] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]
        results = metric.compute(predictions=true_predictions, references=true_labels)
        return {"precision": results["overall_precision"], "recall": results["overall_recall"], "f1": results["overall_f1"], "accuracy": results["overall_accuracy"]}
        
    epochs = 2 if args.smoke_test else 10
    training_args = TrainingArguments(
        output_dir="/kaggle/working/layoutlmv3-avir-kie",
        max_steps=50 if args.smoke_test else -1,
        num_train_epochs=epochs,
        per_device_train_batch_size=8, # v3 nặng hơn nên giảm batch size
        per_device_eval_batch_size=8,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5, # ViT rất nhạy cảm, learning rate phải nhỏ hơn v1
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        processing_class=processor, # Phiên bản mới dùng processing_class thay cho tokenizer
        data_collator=DefaultDataCollator(),
        compute_metrics=compute_metrics,
    )
    
    print("Bắt đầu huấn luyện Proposed Model (LayoutLMv3)...")
    trainer.train()
    trainer.evaluate()
    
    model_dir = "/kaggle/working/layoutlmv3-avir-kie-best"
    trainer.save_model(model_dir)
    processor.save_pretrained(model_dir)
    print(f"Lưu thành công tại {model_dir}")

if __name__ == "__main__":
    main()


In [ ]:
# Chạy huấn luyện
!python /kaggle/working/train_layoutlmv3.py

In [ ]:
# Nén model v3
!zip -r /kaggle/working/layoutlmv3_model_export.zip /kaggle/working/layoutlmv3-avir-kie-best